# Exploring Neural Activity in the Zebrafish Brain
### A hands-on introduction to two-photon calcium imaging

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/VTKLuminance/blob/main/zebrafish_calcium_imaging_colab.ipynb)

---

## What is two-photon calcium imaging?

Neurons become active when they fire electrical impulses called **action potentials**. Each time a neuron fires, calcium ions flood into the cell. By engineering neurons to express a fluorescent protein called **GCaMP** — which glows brighter when calcium is high — we can film brain activity as changes in fluorescence.

**Two-photon microscopy** lets us image hundreds of individual neurons simultaneously in the living brain. Because larval zebrafish (*Danio rerio*) are **transparent**, we can image the *entire* brain at once across multiple depths (z-planes).

## The experiment

A zebrafish was shown visual stimuli — a luminance gradient brighter on either the left or right side — while we recorded fluorescence from ~1000 neurons across 5 imaging depths.

Each trial lasted **60 seconds**:

```
  |--- baseline (10 s) ---|--- stimulus ON (30 s) ---|--- recovery (20 s) ---|
  0                       10                         40                      60 s
```

The two conditions we'll focus on:
- **`lumi_left_strong_dots_off`** — bright patch on the **left**
- **`lumi_right_strong_dots_off`** — bright patch on the **right**

By the end of this notebook you will be able to:
1. Load and explore a real neuroscience dataset from a CSV file
2. Visualise and interpret stimulus-aligned fluorescence traces
3. Quantify neural selectivity with a laterality index
4. Cluster neurons into functional types using machine learning

---

## 0 — Setup

Run these two cells first to install and import everything we need.

In [ ]:
!pip install umap-learn pynrrd -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nrrd
from scipy.stats import zscore
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap

print('Ready!')

### Download the data files

The cell below downloads both data files directly from GitHub — nothing to configure.

In [ ]:
import urllib.request, os

REPO = "https://raw.githubusercontent.com/YOUR_GITHUB_USERNAME/VTKLuminance/main"

files = {
    "lumi_strong_dots_off_traces.csv": f"{REPO}/lumi_strong_dots_off_traces.csv",
    "LightRightmini.nrrd":             f"{REPO}/LightRightmini.nrrd",
}

for filename, url in files.items():
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
    else:
        print(f"{filename} already present")

CSV_PATH  = "lumi_strong_dots_off_traces.csv"
NRRD_PATH = "LightRightmini.nrrd"

# Timing constants — do not change
FPS        = 2.0
N_FRAMES   = 120
TRIAL_TIME = np.arange(N_FRAMES) / FPS   # 0.0, 0.5 … 59.5 s
STIM_ON    = (10, 40)                     # stimulus-on window in seconds

print("Data ready!")

---
## 1 — Loading and understanding the data

The data is stored as a **CSV** (comma-separated values) file — the simplest way to share tabular data. We load it with `pandas`, which gives us a **DataFrame** (like a spreadsheet in Python).

### 1.1 Load the CSV and preview it

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

Each **row** is one neuron in one stimulus condition.  
The first three columns describe the neuron:  
- `z_plane` — imaging depth (z000 = shallowest, z004 = deepest)
- `cell_id` — neuron index within that z-plane
- `condition` — which stimulus was shown

Columns `t000` through `t119` are the **fluorescence values** at each of the 120 time-frames of the trial (0.5 s apart).

### 1.2 What conditions and z-planes are in the dataset?

In [ ]:
print('Conditions:')
for c in df['condition'].unique():
    n = (df['condition'] == c).sum()
    print(f'  {c}  ({n} neurons)')

print('\nZ-planes:')
for z in df['z_plane'].unique():
    n = len(df[(df['z_plane'] == z) & (df['condition'] == df['condition'].unique()[0])])
    print(f'  {z}: {n} neurons')

### 1.3 Extract the trace matrix

For analysis we want a 2-D NumPy array of shape `(n_neurons, 120)` — one row per neuron.  
We get this by:
1. Filtering the DataFrame to one condition
2. Selecting only the `t000`…`t119` columns
3. Converting to a NumPy array

In [ ]:
# Column names for the 120 timepoints
trace_cols = [f't{i:03d}' for i in range(N_FRAMES)]

# Filter to the left and right strong conditions
left_df  = df[df['condition'] == 'lumi_left_strong_dots_off']
right_df = df[df['condition'] == 'lumi_right_strong_dots_off']

# Extract as NumPy arrays: shape (n_neurons, 120)
left_traces  = left_df[trace_cols].values
right_traces = right_df[trace_cols].values

print('left_traces shape:', left_traces.shape)
print('right_traces shape:', right_traces.shape)

---
## 2 — Visualising single-neuron responses

Let's start by looking at individual neurons to get a feel for the data.  
Each curve below shows how one neuron's fluorescence changed over a 60-second trial.  
The yellow band marks the period when the luminance stimulus was on (10–40 s).

In [ ]:
example_cells = [1, 10, 50, 100, 200]

fig, axes = plt.subplots(len(example_cells), 1, figsize=(11, 8), sharex=True)

for ax, idx in zip(axes, example_cells):
    ax.axvspan(STIM_ON[0], STIM_ON[1], color='gold', alpha=0.3)
    ax.plot(TRIAL_TIME, left_traces[idx],  color='#2166AC', lw=1.5, label='Lumi Left')
    ax.plot(TRIAL_TIME, right_traces[idx], color='#D6604D', lw=1.5, label='Lumi Right')
    ax.set_ylabel(f'Neuron {idx}\nF (a.u.)', fontsize=8)

axes[0].legend(fontsize=9, loc='upper right')
axes[-1].set_xlabel('Time (s)', fontsize=11)
fig.suptitle('Example single-neuron trial-averaged responses', fontsize=12)
plt.tight_layout()
plt.show()

> 🔍 **Look at the traces.** Some neurons respond more to Left, some more to Right, and some seem indifferent. This diversity of responses is exactly what we want to characterise!

---

### ✏️ Exercise 2.1 — Pick your own neurons

Change the `example_cells` list to plot **5 neurons of your choice** from **z-plane z002 only**.  

> **Hint:** First get the row indices of z002 neurons: `z002_idx = df[df['z_plane'] == 'z002'].index`  
> Then use those indices to index into `left_traces` and `right_traces`.

In [ ]:
# Get the integer positions (within left_traces) of z002 neurons
# left_df was built from df filtered to the left condition, so reset the index
left_df_reset = left_df.reset_index(drop=True)
z002_positions = left_df_reset[left_df_reset['z_plane'] == ___].index.tolist()  # ← fill in

print(f'Found {len(z002_positions)} z002 neurons')

# Choose 5 of them
chosen = z002_positions[:5]

fig, axes = plt.subplots(5, 1, figsize=(11, 8), sharex=True)
for ax, idx in zip(axes, chosen):
    ax.axvspan(STIM_ON[0], STIM_ON[1], color='gold', alpha=0.3)
    ax.plot(TRIAL_TIME, left_traces[idx],  color='#2166AC', lw=1.5, label='Lumi Left')
    ax.plot(TRIAL_TIME, right_traces[idx], color='#D6604D', lw=1.5, label='Lumi Right')
    ax.set_ylabel(f'idx {idx}', fontsize=8)
axes[0].legend(fontsize=9)
axes[-1].set_xlabel('Time (s)', fontsize=11)
fig.suptitle('Single-neuron responses — z002', fontsize=12)
plt.tight_layout()
plt.show()

---
## 3 — Population average response

Individual neurons are noisy. A powerful approach is to **average across all neurons** to reveal the typical response. We also show the **±1 SEM band** (standard error of the mean) to indicate how consistent the response is across neurons.

$$\text{SEM} = \frac{\text{standard deviation}}{\sqrt{n}}$$

In [ ]:
def mean_sem(traces):
    """Return population mean and SEM across rows (neurons)."""
    m  = traces.mean(axis=0)
    se = traces.std(axis=0) / np.sqrt(traces.shape[0])
    return m, se

left_m,  left_se  = mean_sem(left_traces)
right_m, right_se = mean_sem(right_traces)

fig, ax = plt.subplots(figsize=(10, 4))
ax.axvspan(STIM_ON[0], STIM_ON[1], color='gold', alpha=0.25, label='Stimulus ON')
ax.axvline(STIM_ON[0], color='gray', lw=0.8, ls='--')
ax.axvline(STIM_ON[1], color='gray', lw=0.8, ls='--')

for m, se, color, label in [
    (left_m,  left_se,  '#2166AC', f'Lumi Left  (n={left_traces.shape[0]})'),
    (right_m, right_se, '#D6604D', f'Lumi Right (n={right_traces.shape[0]})'),
]:
    ax.plot(TRIAL_TIME, m, color=color, lw=2, label=label)
    ax.fill_between(TRIAL_TIME, m - se, m + se, color=color, alpha=0.25)

ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Fluorescence (a.u.)', fontsize=11)
ax.set_title('Population mean ± SEM — all z-planes', fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim(0, TRIAL_TIME[-1])
plt.tight_layout()
plt.show()

> 🔍 **Observation:** The two conditions produce *opposite* average responses. Can you explain why a left-luminance stimulus might *decrease* fluorescence in some neurons while a right-luminance stimulus *increases* it?

---

### ✏️ Exercise 3.1 — Population average per z-plane

The cells above come from 5 different imaging depths. Do all depths show the same population response, or do some depths look different?

Complete the code below to plot the population mean for **each z-plane separately** in a row of subplots.

In [ ]:
z_planes = ['z000', 'z001', 'z002', 'z003', 'z004']
left_df_reset  = left_df.reset_index(drop=True)
right_df_reset = right_df.reset_index(drop=True)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5), sharey=False)

for ax, z in zip(axes, z_planes):
    # Get row positions belonging to this z-plane
    idx = left_df_reset[left_df_reset['z_plane'] == ___].index  # ← fill in

    left_z  = left_traces[idx]
    right_z = right_traces[idx]

    lm, ls = mean_sem(left_z)
    rm, rs = mean_sem(___)

    ax.axvspan(STIM_ON[0], STIM_ON[1], color='gold', alpha=0.25)
    ax.plot(TRIAL_TIME, lm, color='#2166AC', lw=1.5, label=f'Left (n={len(idx)})')
    ax.fill_between(TRIAL_TIME, lm - ls, lm + ls, color='#2166AC', alpha=0.2)
    ax.plot(TRIAL_TIME, rm, color='#D6604D', lw=1.5, label=f'Right')
    ax.fill_between(TRIAL_TIME, rm - rs, rm + rs, color='#D6604D', alpha=0.2)
    ax.set_title(z, fontsize=10)
    ax.set_xlabel('Time (s)', fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlim(0, TRIAL_TIME[-1])

axes[0].set_ylabel('F (a.u.)', fontsize=10)
fig.suptitle('Population mean per z-plane', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4 — Quantifying selectivity: the Laterality Index

Looking at traces is informative, but we want a single number that captures *how much* each neuron prefers left vs right. We define a **Laterality Index (LI)**:

$$\text{LI} = \frac{\bar{F}_{\text{right}} - \bar{F}_{\text{left}}}{|\bar{F}_{\text{right}}| + |\bar{F}_{\text{left}}|}$$

where $\bar{F}$ is the **mean fluorescence during the stimulus-ON window** (10–40 s).

| LI value | Interpretation |
|----------|----------------|
| Close to +1 | Strongly prefers **right** luminance |
| Close to −1 | Strongly prefers **left** luminance |
| Close to 0  | No directional preference |

In [ ]:
# Stimulus-ON frames: 10 s × 2 fps = frame 20 through frame 79
stim_frames = slice(20, 80)

left_stim  = left_traces[:,  stim_frames].mean(axis=1)   # (n_neurons,)
right_stim = right_traces[:, stim_frames].mean(axis=1)

LI = (right_stim - left_stim) / (np.abs(right_stim) + np.abs(left_stim) + 1e-9)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(LI, bins=50, color='mediumpurple', edgecolor='white', lw=0.5)
ax.axvline(0, color='black', lw=1.5, ls='--', label='LI = 0 (no preference)')
ax.set_xlabel('Laterality Index', fontsize=11)
ax.set_ylabel('Number of neurons', fontsize=11)
ax.set_title('Distribution of luminance preference across all neurons', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'Neurons preferring right  (LI >  0.1): {(LI >  0.1).sum()}')
print(f'Neurons preferring left   (LI < -0.1): {(LI < -0.1).sum()}')
print(f'Neurons non-selective (|LI| ≤ 0.1):    {(np.abs(LI) <= 0.1).sum()}')

---

### ✏️ Exercise 4.1 — Response magnitude vs selectivity

The LI tells us *which side* a neuron prefers, but not how strongly it responds.  
Compute a **response magnitude** for each neuron:

$$\text{magnitude} = \max\left(|\bar{F}_{\text{left,stim}} - \bar{F}_{\text{left,base}}|,\; |\bar{F}_{\text{right,stim}} - \bar{F}_{\text{right,base}}|\right)$$

Then make a **scatter plot** of LI (x-axis) vs magnitude (y-axis), coloured by LI.  
What pattern do you expect to see?

In [ ]:
baseline_frames = slice(0, 20)   # 0–10 s

left_base  = left_traces[:,  baseline_frames].mean(axis=1)
right_base = right_traces[:, baseline_frames].mean(axis=1)

left_resp  = np.abs(left_stim  - ___)
right_resp = np.abs(right_stim - ___)
magnitude  = np.maximum(left_resp, right_resp)

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(___, magnitude, s=6, alpha=0.4,   # ← x-axis: LI
                c=LI, cmap='RdBu_r', vmin=-1, vmax=1)
ax.axvline(0, color='black', lw=1, ls='--')
plt.colorbar(sc, ax=ax, label='Laterality Index')
ax.set_xlabel('Laterality Index', fontsize=11)
ax.set_ylabel('Response magnitude (a.u.)', fontsize=11)
ax.set_title('Selectivity vs response strength', fontsize=12)
plt.tight_layout()
plt.show()

---
## 5 — Functional clustering

So far we have treated all neurons as belonging to one population. But the brain contains distinct **functional cell types** — groups of neurons with similar response profiles. We can discover them automatically with **unsupervised machine learning**.

### Our pipeline

```
Raw traces
    ↓  z-score each trace (normalise scale)
Feature matrix  (n_neurons × 240)
    ↓  PCA  (reduce noise, keep 95% variance)
PC scores  (n_neurons × ~80)
    ↓  K-means  (assign each neuron to a cluster)
Cluster labels  (n_neurons,)
    ↓  UMAP  (project to 2-D for visualisation)
2-D embedding
```

### 5.1 Build the feature matrix

We **z-score** each trace so that neurons with large vs small baseline fluorescence contribute equally. Then we concatenate the left and right traces into a single 240-dimensional vector per neuron.

In [ ]:
# Z-score each neuron's trace (axis=1 → across time)
left_z  = np.array([zscore(t) for t in left_traces])    # (n_neurons, 120)
right_z = np.array([zscore(t) for t in right_traces])

# Concatenate: left response + right response → 240-dim feature vector
features = np.hstack([left_z, right_z])                  # (n_neurons, 240)
print('Feature matrix shape:', features.shape)

### 5.2 PCA — reduce dimensionality

240 dimensions is too many for k-means to work well (the **curse of dimensionality**). PCA finds the axes of maximum variance and projects the data onto them, keeping only as many as needed to explain 95% of the variance.

In [ ]:
pca = PCA(n_components=0.95, svd_solver='full')
X_pca = pca.fit_transform(features)
print(f'PCA: {features.shape[1]} dimensions → {X_pca.shape[1]} (95% variance retained)')

# Scree plot
fig, ax = plt.subplots(figsize=(6, 3))
cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
ax.plot(cumvar, color='steelblue', lw=2)
ax.axhline(95, color='crimson', ls='--', lw=1, label='95% threshold')
ax.set_xlabel('PCA component', fontsize=10)
ax.set_ylabel('Cumulative variance (%)', fontsize=10)
ax.set_title('How many PCA components do we need?', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 5.3 Choose k with the silhouette score

K-means requires us to specify **k** (the number of clusters) in advance. The **silhouette score** measures how well-separated the clusters are — higher is better. We try k = 2 to 8 and pick the best.

In [ ]:
k_range    = range(2, 9)
sil_scores = []

for k in k_range:
    labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X_pca)
    score  = silhouette_score(X_pca, labels)
    sil_scores.append(score)
    print(f'  k={k}  silhouette={score:.4f}')

best_k = list(k_range)[int(np.argmax(sil_scores))]
print(f'\n→ Best k = {best_k}')

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(list(k_range), sil_scores, 'o-', color='steelblue', lw=2)
ax.axvline(best_k, color='crimson', ls='--', lw=1.5, label=f'Best k={best_k}')
ax.set_xlabel('k (number of clusters)', fontsize=10)
ax.set_ylabel('Silhouette score', fontsize=10)
ax.set_title('Choosing k', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

> 💡 **Note:** Silhouette scores in calcium imaging are typically low (0.1–0.2). This doesn't mean the analysis is wrong — it means neurons vary along a *continuum* of response types rather than falling into perfectly separated groups. The clusters still capture the dominant functional trends.

### 5.4 Fit the final model and visualise clusters

In [ ]:
km = KMeans(n_clusters=best_k, n_init=30, random_state=42)
cluster_labels = km.fit_predict(X_pca)

for k in range(best_k):
    print(f'Cluster {k}: {(cluster_labels == k).sum()} neurons')

In [ ]:
cmap = plt.cm.tab10

ncols = min(best_k, 4)
nrows = int(np.ceil(best_k / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.5*nrows),
                          sharey=False, squeeze=False)
fig.suptitle('Mean response ± SEM per cluster', fontsize=13)

for k in range(best_k):
    ax   = axes[k // ncols][k % ncols]
    mask = cluster_labels == k
    ax.axvspan(STIM_ON[0], STIM_ON[1], color='gold', alpha=0.25)
    ax.axvline(STIM_ON[0], color='gray', lw=0.7, ls='--')
    ax.axvline(STIM_ON[1], color='gray', lw=0.7, ls='--')

    for traces, color, label in [
        (left_traces[mask],  '#2166AC', 'Lumi Left'),
        (right_traces[mask], '#D6604D', 'Lumi Right'),
    ]:
        m, se = mean_sem(traces)
        ax.plot(TRIAL_TIME, m, color=color, lw=1.8, label=label)
        ax.fill_between(TRIAL_TIME, m - se, m + se, color=color, alpha=0.2)

    ax.set_title(f'Cluster {k}  (n={mask.sum()})', fontsize=10,
                 color=cmap(k), fontweight='bold')
    ax.set_xlabel('Time (s)', fontsize=9)
    ax.set_ylabel('F (a.u.)', fontsize=9)
    ax.legend(fontsize=8)
    ax.set_xlim(0, TRIAL_TIME[-1])

for idx in range(best_k, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

plt.tight_layout()
plt.show()

### 5.5 UMAP — see the clusters in 2-D

UMAP compresses the ~80 PCA dimensions down to 2 for visualisation, trying to preserve which neurons are *similar* to each other. Each dot is one neuron.

In [ ]:
print('Running UMAP (~30 s)...')
reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42)
X_umap  = reducer.fit_transform(X_pca)

fig, ax = plt.subplots(figsize=(6, 5))
for k in range(best_k):
    mask = cluster_labels == k
    ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
               s=7, alpha=0.5, color=cmap(k),
               label=f'Cluster {k}  (n={mask.sum()})')
ax.set_xlabel('UMAP 1', fontsize=10)
ax.set_ylabel('UMAP 2', fontsize=10)
ax.set_title(f'UMAP — {best_k} functional clusters', fontsize=11)
ax.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.show()

### 5.6 Heatmap — all neurons sorted by cluster

A heatmap lets us see the response pattern of every neuron at once. Each row is one neuron (z-scored), sorted by cluster.

In [ ]:
sort_order = np.argsort(cluster_labels)
n_total    = len(cluster_labels)
hm_left  = np.array([zscore(left_traces[i])  for i in sort_order])
hm_right = np.array([zscore(right_traces[i]) for i in sort_order])

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, data, title in zip(axes, [hm_left, hm_right],
                            ['Lumi Left', 'Lumi Right']):
    im = ax.imshow(data, aspect='auto', cmap='RdBu_r', vmin=-2.5, vmax=2.5,
                   origin='upper',
                   extent=[TRIAL_TIME[0], TRIAL_TIME[-1], n_total, 0])
    ax.axvline(STIM_ON[0], color='white', lw=1, ls='--')
    ax.axvline(STIM_ON[1], color='white', lw=1, ls='--')
    ax.set_xlabel('Time (s)', fontsize=10)
    ax.set_ylabel('Neuron (sorted by cluster)', fontsize=10)
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, label='z-score', shrink=0.8)
    boundaries = np.where(np.diff(cluster_labels[sort_order]))[0] + 1
    for b in boundaries:
        ax.axhline(b, color='black', lw=1)

fig.suptitle(f'All neurons sorted by cluster ({best_k} clusters)', fontsize=12)
plt.tight_layout()
plt.show()

---

### ✏️ Exercise 5.1 — Try a different k

The silhouette score suggested k=2, but you might want to explore finer-grained types.  
Re-run k-means with **k = 4** and plot the cluster mean traces.  
Do the additional clusters reveal any new response patterns?

In [ ]:
km4 = KMeans(n_clusters=___, n_init=30, random_state=42)  # ← fill in k
labels4 = km4.fit_predict(___)

fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), sharey=False)
for k, ax in enumerate(axes):
    mask = labels4 == k
    ax.axvspan(STIM_ON[0], STIM_ON[1], color='gold', alpha=0.25)
    for traces, color, label in [
        (left_traces[mask],  '#2166AC', 'Left'),
        (right_traces[mask], '#D6604D', 'Right'),
    ]:
        m, se = mean_sem(traces)
        ax.plot(TRIAL_TIME, m, color=color, lw=1.8, label=label)
        ax.fill_between(TRIAL_TIME, m - se, m + se, color=color, alpha=0.2)
    ax.set_title(f'Cluster {k}  (n={mask.sum()})', fontsize=10, color=cmap(k), fontweight='bold')
    ax.set_xlabel('Time (s)', fontsize=9)
    ax.legend(fontsize=8)
    ax.set_xlim(0, TRIAL_TIME[-1])
axes[0].set_ylabel('F (a.u.)', fontsize=9)
fig.suptitle('k=4 cluster mean traces', fontsize=12)
plt.tight_layout()
plt.show()

### ✏️ Exercise 5.2 — Colour UMAP by laterality index

Instead of colouring by cluster label, colour each point by its **laterality index** from Section 4.  
Does the LI gradient align with the UMAP geometry?  
What does this tell you about how the brain organises left/right selectivity?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(X_umap[:, 0], X_umap[:, 1],
                c=___,              # ← colour by LI (computed in Section 4)
                cmap='RdBu_r', vmin=-1, vmax=1,
                s=7, alpha=0.6)
plt.colorbar(sc, ax=ax, label='Laterality Index (blue=Left, red=Right)')
ax.set_xlabel('UMAP 1', fontsize=10)
ax.set_ylabel('UMAP 2', fontsize=10)
ax.set_title('UMAP coloured by laterality index', fontsize=11)
plt.tight_layout()
plt.show()

---
## 6 — Pixel-PCA of the raw imaging movie

So far we've worked with pre-extracted cell traces. But we can apply PCA **directly to the raw movie pixels** — no cell detection required. This gives an unbiased overview of what activity patterns exist in the image.

The `LightRightmini.nrrd` file is a downsampled 2-D movie of one brain slice: **81 × 90 pixels × 120 timeframes**. Each pixel records the fluorescence of a small patch of tissue over the 60-second trial.

### The idea

We reshape the movie into a **pixel matrix** of shape `(n_pixels, n_timeframes)` — one row per pixel. PCA then finds the **dominant spatiotemporal patterns**:

- The **PC scores** reshaped back to 81 × 90 are **spatial maps** — *where* in the image each pattern is expressed.
- The **PC time courses** show *when* each pattern is active.

Red pixels in a spatial map are positively weighted (active when the time course is high); blue pixels are negatively weighted.

### 6.1 Load the movie and display the mean image

In [ ]:
movie, header = nrrd.read(NRRD_PATH)
nx, ny, nt = movie.shape
print(f'Movie shape: {nx} × {ny} pixels, {nt} frames  ({nt/FPS:.0f} s at {FPS} fps)')

mean_img = movie.mean(axis=2)   # average over time → (nx, ny)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(mean_img.T, cmap='gray', origin='lower')
ax.set_title('Mean fluorescence image (averaged over all frames)', fontsize=11)
ax.set_xlabel('X (pixels)', fontsize=10)
ax.set_ylabel('Y (pixels)', fontsize=10)
plt.tight_layout()
plt.show()

> 🧠 The bright spots are individual neurons expressing GCaMP. The dimmer background is neuropil (axons and dendrites).

### 6.2 Reshape and z-score

We **z-score** each pixel's time series so that bright and dim pixels contribute equally — otherwise PCA would just find the brightest pixels rather than the most *informative* ones.

In [ ]:
# Reshape: (nx, ny, nt) → (n_pixels, nt)
X_pixels = movie.reshape(-1, nt).astype(np.float64)
print('Pixel matrix shape:', X_pixels.shape, ' → (n_pixels, n_timeframes)')

# Z-score each pixel's time series across frames
X_z = (X_pixels - X_pixels.mean(axis=1, keepdims=True)) / \
      (X_pixels.std(axis=1,  keepdims=True) + 1e-6)

### 6.3 Apply PCA and inspect the scree plot

In [ ]:
N_COMPONENTS = 10
pca_px      = PCA(n_components=N_COMPONENTS)
scores      = pca_px.fit_transform(X_z)   # (n_pixels, 10) — spatial weights
timecourses = pca_px.components_          # (10, 120)       — time courses
var_px      = pca_px.explained_variance_ratio_ * 100

print('Variance explained per PC:')
for i, v in enumerate(var_px):
    print(f'  PC{i+1}: {v:.1f}%')

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(1, N_COMPONENTS + 1), var_px, color='steelblue', edgecolor='white')
ax.set_xlabel('PC', fontsize=11)
ax.set_ylabel('Variance explained (%)', fontsize=11)
ax.set_title('Pixel-PCA scree plot', fontsize=12)
ax.set_xticks(range(1, N_COMPONENTS + 1))
plt.tight_layout()
plt.show()

### 6.4 Spatial maps and time courses

Each PC gives us a paired **spatial map** and **time course**. The yellow band marks the stimulus-on window.

In [ ]:
import matplotlib.gridspec as gridspec

N_SHOW = 6
fig = plt.figure(figsize=(15, 9))
fig.suptitle('Pixel-PCA: spatial maps and time courses', fontsize=13, y=1.01)
outer = gridspec.GridSpec(N_SHOW, 2, figure=fig,
                          width_ratios=[1, 2.5], hspace=0.6, wspace=0.3)

for i in range(N_SHOW):
    spatial_map = scores[:, i].reshape(nx, ny)
    tc          = timecourses[i]
    vmax        = np.percentile(np.abs(spatial_map), 98)

    ax_img = fig.add_subplot(outer[i, 0])
    im = ax_img.imshow(spatial_map.T, cmap='RdBu_r',
                       vmin=-vmax, vmax=vmax, origin='lower')
    ax_img.set_title(f'PC{i+1}  ({var_px[i]:.1f}%)', fontsize=9, pad=2)
    ax_img.axis('off')
    plt.colorbar(im, ax=ax_img, fraction=0.046, pad=0.04)

    ax_tc = fig.add_subplot(outer[i, 1])
    ax_tc.axvspan(STIM_ON[0], STIM_ON[1], color='gold', alpha=0.25)
    ax_tc.axvline(STIM_ON[0], color='gray', lw=0.7, ls='--')
    ax_tc.axvline(STIM_ON[1], color='gray', lw=0.7, ls='--')
    ax_tc.axhline(0, color='gray', lw=0.5)
    ax_tc.plot(TRIAL_TIME, tc, color='steelblue', lw=1.5)
    ax_tc.set_xlim(0, TRIAL_TIME[-1])
    ax_tc.set_ylabel(f'PC{i+1}', fontsize=8)
    if i < N_SHOW - 1:
        ax_tc.set_xticklabels([])
    else:
        ax_tc.set_xlabel('Time (s)', fontsize=9)

plt.show()

> 🔍 **What to look for:**
> - PCs whose **time course changes inside the yellow band** (10–40 s) are **stimulus-driven** patterns.
> - PCs with slow monotone drifts reflect **photobleaching** — a gradual loss of fluorescence that is a common imaging artefact.
> - Check whether the spatial map of a stimulus-responsive PC looks like individual cell bodies or a more diffuse region.

---

### ✏️ Exercise 6.1 — Identify stimulus-responsive PCs

Look at the time courses above. Which PCs show a clear change locked to the stimulus-on period?  
Fill in their indices (0-based) below and plot just their spatial maps side by side.

In [ ]:
# ← Fill in the PC indices (0-based) that look stimulus-responsive
responsive_pcs = [___, ___]

fig, axes = plt.subplots(1, len(responsive_pcs), figsize=(5 * len(responsive_pcs), 4))
if len(responsive_pcs) == 1:
    axes = [axes]

for ax, pc_idx in zip(axes, responsive_pcs):
    spatial_map = scores[:, pc_idx].reshape(nx, ny)
    vmax = np.percentile(np.abs(spatial_map), 98)
    im = ax.imshow(spatial_map.T, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
    ax.set_title(f'PC{pc_idx+1}  ({var_px[pc_idx]:.1f}%)', fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('Stimulus-responsive PC spatial maps', fontsize=12)
plt.tight_layout()
plt.show()

### ✏️ Exercise 6.2 — Reconstruct the movie from k PCs

PCA gives a compressed representation. We can reconstruct an approximation of the movie using only the first `k` components:

$$\hat{X} = \text{scores}_{:,\,0:k} \;\cdot\; \text{timecourses}_{0:k,\,:}$$

Complete the code below to compare one original frame with its `k`-PC reconstruction.  
Try `k = 1, 3, 5, 10` — how many components does it take to recover the main structure?

In [ ]:
k_reconstruct = 3    # ← try 1, 3, 5, 10
frame_index   = 30   # ← frame to display (0–119)

# Reconstruct: scores[:, :k] @ timecourses[:k, :]
X_recon = scores[:, :k_reconstruct] @ timecourses[:k_reconstruct, :]  # (n_pixels, 120)

frame_orig  = X_z[:, frame_index].reshape(nx, ny)
frame_recon = X_recon[:, ___].reshape(nx, ny)   # ← fill in frame index

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, img, title in zip(axes,
                           [frame_orig, frame_recon],
                           [f'Original (z-scored) — frame {frame_index}',
                            f'Reconstructed with k={k_reconstruct} PCs']):
    vmax = np.percentile(np.abs(img), 98)
    ax.imshow(img.T, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## 7 — Reflection questions

Write your answers in the text cell below each question.

**Q1.** Why do we z-score each neuron's trace before clustering? What would happen if we skipped this step?

**Q2.** The silhouette scores were all below 0.2. Does this mean the clustering is unreliable, or does it tell us something genuine about how neurons are organised?

**Q3.** Describe in your own words what Cluster 0 represents biologically. What kind of computation might a neuron be performing if it is excited by right luminance but suppressed by left luminance?

**Q4.** We only used two stimulus conditions (`lumi_left_strong` and `lumi_right_strong`) for clustering. What other conditions from this dataset would you add to better characterise cell types, and why?

**Q5.** *(Challenge)* How would you test whether the clusters you found are reproducible? What would you do to check that they aren't just an artefact of the random initialisation of k-means?

---
## 🎉 Well done!

You've worked through the core analysis pipeline of modern systems neuroscience:
loading real data → visualising responses → quantifying selectivity → unsupervised clustering.

The same tools scale from ~1000 neurons (as here) to millions, and from zebrafish to mice and humans.

**Further reading**
- Ahrens et al. (2013) *Whole-brain functional imaging at cellular resolution.* Nature Methods.
- Portugues et al. (2014) *Whole-brain activity maps reveal stereotyped networks for visuomotor behavior.* Neuron.
- McInnes et al. (2018) *UMAP: Uniform Manifold Approximation and Projection.* arXiv:1802.03426.